## 로컬 환경 설정

로컬 Jupyter나 VS Code에서 실행할 때 현재 경로의 부모를 탐색해 저장소 루트를 찾는다. 로컬 dependency는 notebook 설치 셀보다 `uv sync`로 맞춘 환경을 우선한다.


In [1]:
import os
import sys
from pathlib import Path

# torch import 전에 반드시 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

# 절대 경로로 직접 지정
LOCAL_PROJECT_PATH = Path("/home/undergraduate/20231372_TY/Pytorch/VLM")
os.chdir(LOCAL_PROJECT_PATH)

if str(LOCAL_PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(LOCAL_PROJECT_PATH))

print("Project root:", os.getcwd())
# 출력: /home/undergraduate/20231372_TY/Pytorch/VLM  이 나와야 정상

Project root: /home/undergraduate/20231372_TY/Pytorch/VLM


## 공통 import

노트북 전체에서 사용하는 외부 패키지와 저장소 공용 모듈을 한 번만 불러온다.


In [2]:
import os
import json
from dataclasses import dataclass
from pathlib import Path

MPLCONFIGDIR = Path(os.environ.get("MPLCONFIGDIR", "/tmp/matplotlib"))
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import torch
import transformers
from PIL import Image
from torch.utils.data import Dataset

from utils.caption_metrics import compute_caption_metrics

from utils.device_utils import (
    get_device_info,
    release_cuda_memory,
    resolve_device,
)

from utils.logger_utils import configure_logger, get_logger

from utils.smolvlm_utils import (
    align_model_generation_config_with_tokenizer,
    generate_caption_splits,
    load_model_and_processor,
    resolve_auto_model_class,
    resolve_torch_dtype,
)

from utils.training_utils import (
    build_config_payload,
    save_json,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from utils.caption_metrics import safe_compute_single_caption_scores
from utils.lora_finetuning import train_and_evaluate_lora_rank

/home/undergraduate/20231372_TY/envs/smolvlm_lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 실행 설정과 상수

모델, prompt, output 경로, LoRA target module, Trainer hyperparameter를 고정한다. 기본 실행은 rank 4/8/16 sweep으로 LoRA adapter 크기에 따른 비용과 성능 변화를 비교한다.


In [3]:
# 실행 설정과 상수
# COCO가 아니라 NurViD에서 추출한 의료 이미지 + 직접 작성한 GT caption을 사용한다.

LOGGER = get_logger("nurvid_smolvlm_lora")

@dataclass(slots=True)
class LoRAFinetuneConfig:
    model_name: str = "HuggingFaceTB/SmolVLM-256M-Instruct"
    prompt: str = "Describe this medical scene in one concise English sentence."

    # rank 3개로 비교
    ranks: tuple[int, ...] = (4, 8, 16)

    target_modules: tuple[str, ...] = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )

    seed: int = 42
    device: str = "cuda"

    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4  # 1 → 4 (effective batch=4)

    # 데이터 30개니까 epoch 늘림
    num_train_epochs: int = 10            # 1 → 10
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1            # 0.03 → 0.1
    logging_steps: int = 1
    max_new_tokens: int = 64             # 24 → 64

    image_dir: Path = Path("dataset/selected_images")
    caption_path: Path = Path("dataset/nurvid_captions.jsonl")

    data_dir: Path = Path("data/nurvid-lora-finetuning")
    output_dir: Path = Path("model/nurvid-lora-finetuning")

In [4]:
config_lora_ft = LoRAFinetuneConfig()

config_lora_ft.data_dir.mkdir(parents=True, exist_ok=True)
config_lora_ft.output_dir.mkdir(parents=True, exist_ok=True)

print("Image dir:", config_lora_ft.image_dir)
print("Caption path:", config_lora_ft.caption_path)
print("Output dir:", config_lora_ft.output_dir)

Image dir: dataset/selected_images
Caption path: dataset/nurvid_captions.jsonl
Output dir: model/nurvid-lora-finetuning


## LoRA 학습 준비

LoRA fine-tuning 설정, device 정보, 저장 경로를 준비한다. = 학습을 시작하기 전 환경을 준비하고 설정을 기록하는 코드


In [5]:
# LoRA 학습 준비

configure_logger()
config_lora_ft = LoRAFinetuneConfig()

# 디렉토리 생성
config_lora_ft.data_dir.mkdir(parents=True, exist_ok=True)
config_lora_ft.output_dir.mkdir(parents=True, exist_ok=True)

# 이미지 / 캡션 파일 확인
image_files = sorted([
    p for p in config_lora_ft.image_dir.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

if not config_lora_ft.image_dir.exists():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {config_lora_ft.image_dir}")

if not config_lora_ft.caption_path.exists():
    raise FileNotFoundError(f"캡션 파일이 없습니다: {config_lora_ft.caption_path}")

LOGGER.info(
    "Starting NurViD LoRA run | model=%s images=%s ranks=%s output_dir=%s",
    config_lora_ft.model_name,
    len(image_files),
    config_lora_ft.ranks,
    config_lora_ft.output_dir,
)

# GPU 확인
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다. 현재 torch.cuda.is_available() = False 입니다.")

device = resolve_device(config_lora_ft.device)

device_info = get_device_info(str(device))
save_json(
    config_lora_ft.output_dir / "device_info.json",
    {
        **device_info,
        "device": str(device_info["device"]),
    },
)

print("Device:", device)
print("Image count:", len(image_files))
print("Caption file:", config_lora_ft.caption_path)

01:48:50 |   22 | INFO     | Starting NurViD LoRA run | model=HuggingFaceTB/SmolVLM-256M-Instruct images=35 ranks=(4, 8, 16) output_dir=model/nurvid-lora-finetuning
Device: cuda
Image count: 35
Caption file: dataset/nurvid_captions.jsonl


In [6]:
# selected_images 안의 이미지 목록으로 caption 템플릿 만들기
# ✅ 파일이 이미 존재하면 실행하지 않음

caption_path = Path("dataset/nurvid_captions.jsonl")
image_dir = Path("dataset/selected_images")

if caption_path.exists():
    print(f"[SKIP] 캡션 파일이 이미 존재합니다: {caption_path}")
    print("[INFO] 기존 캡션 파일을 유지합니다.")
else:
    image_files = sorted([
        p.name for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

    with open(caption_path, "w", encoding="utf-8") as f:
        for image_name in image_files:
            row = {
                "image": image_name,
                "caption": "A nurse performing a medical action in a hospital setting."
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"[OK] Caption template saved: {caption_path}")

[SKIP] 캡션 파일이 이미 존재합니다: dataset/nurvid_captions.jsonl
[INFO] 기존 캡션 파일을 유지합니다.


In [7]:
# NurViD 데이터 split 준비

def load_nurvid_caption_rows(caption_path: Path, image_dir: Path):
    rows = []

    with open(caption_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            image_path = image_dir / item["image"]

            if not image_path.exists():
                print(f"[WARN] 이미지 파일 없음: {image_path}")
                continue

            rows.append({
                "image": Image.open(image_path).convert("RGB"),
                "caption": item["caption"],
                "image_path": str(image_path),
                "image_name": item["image"],
            })

    return rows


all_rows = load_nurvid_caption_rows(
    caption_path=config_lora_ft.caption_path,
    image_dir=config_lora_ft.image_dir,
)

print("전체 NurViD 이미지 개수:", len(all_rows))

if len(all_rows) == 0:
    raise ValueError("사용할 이미지가 없습니다. selected_images와 nurvid_captions.jsonl을 확인하세요.")

# 데이터가 적으므로 80:20으로 간단 split
split_idx = max(1, int(len(all_rows) * 0.8))

train_rows = all_rows[:split_idx]
val_rows = all_rows[split_idx:]

# 이미지 수가 너무 적어서 validation이 비면 train 일부를 val로 재사용
if len(val_rows) == 0:
    val_rows = train_rows[:1]

# Before/After 비교용 샘플
comparison_samples = all_rows[: min(4, len(all_rows))]

# metric 계산용 샘플
metric_samples = all_rows

manifest = {
    "train": [row["image_name"] for row in train_rows],
    "val": [row["image_name"] for row in val_rows],
    "metric": [row["image_name"] for row in metric_samples],
    "comparison": [row["image_name"] for row in comparison_samples],
}

save_json(
    config_lora_ft.data_dir / "split_manifest.json",
    manifest,
)

print("train rows:", len(train_rows))
print("val rows:", len(val_rows))
print("metric samples:", len(metric_samples))
print("comparison samples:", len(comparison_samples))

전체 NurViD 이미지 개수: 35
train rows: 28
val rows: 7
metric samples: 35
comparison samples: 4


In [8]:
# zero-shot 추론 준비(LoRA 학습 전 기준점을 만들기 위해 모델을 로드하는 코드)

# zero-shot baseline 생성을 위해 base SmolVLM과 processor를 먼저 로드합니다.
# auto_model_class와 torch_dtype는 이후 rank별 LoRA 학습 helper가 같은 model class와 dtype 정책을 재사용하도록 보관합니다.

# 이 gpu에서 모델을 어떤 정밀도로 올릴지 자동으로 결정
# auto로 두면 T4에 맞게 자동으로 FP16을 선택
torch_dtype = resolve_torch_dtype(device=device, torch_dtype=torch.float16)

# SmolVLM 같은 멀티모달 모델을 로드할 때 어떤 클래스를 써야 하는지 자동으로 찾아주는 코드
# SmolVLM은 이미지 + 텍스트를 처리하는 멀티모달이므로 AutoModelForImageTextToText
# 이 값을 저장해두는 이유는 나중에 rank별 LoRA 학습할 때도 같은 클래스를 재사용하기 위함
auto_model_class = resolve_auto_model_class(transformers)

# SmolVLM 모델과 프로세서를 한 번에 로드하는 함수
zero_shot_bundle = load_model_and_processor(
    model_id=config_lora_ft.model_name, # 어떤 모델을 불러올지 지정(SmolVLM-256M-Instruct)
    device=device, # 모델을 어디에 올릴지 지정
    torch_dtype=torch_dtype, # FP16
    prefer_flash_attention=False, # FlashAttention은 Attention 연산을 더 빠르게 하는 기술인데, T4에서는 지원되지 않아 끄는 것
)
'''
반환값 : 모델과 프로세서를 딕셔너리로 묶어서 반환해준다.
  zero_shot_bundle = {
      "model":     실제 신경망,이미지를 보고 캡션을 생성
      "processor": 이미지/텍스트를 모델 입력 형식으로 변환
  }
'''

# 반환값에서 각각 꺼낸다
processor = zero_shot_bundle["processor"]
zero_shot_model = zero_shot_bundle["model"]

# 모델의 생성 설정과 토크나이저를 서로 일치시키는 코드
# 모델과 토크나이저의 문장 끝 토큰을 일치시킨다.

'''
맞추기 전                    맞춘 후
─────────────   ──────────────
모델:  eos = 2               모델:  eos = 32000
토크나이저: eos = 32000  →   토크나이저: eos = 32000
→ 캡션이 안 끝남              → 캡션이 정상 종료
'''

align_model_generation_config_with_tokenizer(zero_shot_model, processor.tokenizer)

print("model loaded:", type(zero_shot_model))
print("processor loaded:", type(processor))
print("torch dtype:", torch_dtype)
print("device:", device)


model loaded: <class 'transformers.models.idefics3.modeling_idefics3.Idefics3ForConditionalGeneration'>
processor loaded: <class 'transformers.models.idefics3.processing_idefics3.Idefics3Processor'>
torch dtype: torch.float16
device: cuda


## zero-shot 추론 실행과 평가

LoRA 학습 전 기준 caption과 metric을 만든다. = 아래의 코드는 아무것도 학습하지 않은 상태에서 모델이 캡션을 얼마나 잘 생성하는지 측정하는 코드

> 점수 계산 zero_shot_scores

```
모델 생성: "A dog is running in a field"
정답:      "A brown dog runs on the grass"
        ↓
BLEU   = 0.32
ROUGE  = 0.45
METEOR = 0.38
CIDEr  = 0.71
```

safe가 붙은 이유는 점수 계산 중 오류가 나도 프로그램이 멈추지 않게 처리했기 때문이다



In [9]:
# zero-shot 추론 실행과 평가

zero_shot_outputs = generate_caption_splits(
    label="Zero-shot",
    model=zero_shot_model,
    processor=processor,
    sample_splits={
        "comparison": comparison_samples,
        "metric": metric_samples,
    },
    device=device,
    prompt=config_lora_ft.prompt,
    max_new_tokens=config_lora_ft.max_new_tokens,
    batch_size=config_lora_ft.per_device_eval_batch_size,
    logger=LOGGER,
)

zero_shot_comparison_predictions = zero_shot_outputs["comparison"]
zero_shot_metric_predictions = zero_shot_outputs["metric"]

# metric_samples에서 GT caption만 꺼내기
references = [sample["caption"] for sample in metric_samples]

# 점수 계산
zero_shot_scores = compute_caption_metrics(
    predictions=zero_shot_metric_predictions,
    references=references,
)

# 점수 저장
save_json(
    config_lora_ft.output_dir / "zero_shot_caption_scores.json",
    zero_shot_scores,
)

print("Zero-shot scores:", zero_shot_scores)

# 생성 결과도 확인용으로 저장
zero_shot_result_rows = []

for sample, pred in zip(metric_samples, zero_shot_metric_predictions):
    zero_shot_result_rows.append({
        "image_name": sample["image_name"],
        "gt_caption": sample["caption"],
        "zero_shot_prediction": pred,
    })

save_json(
    config_lora_ft.output_dir / "zero_shot_predictions.json",
    zero_shot_result_rows,
)

print("Zero-shot predictions saved.")

01:48:55 |  366 | INFO     | Running zero-shot caption generation for comparison samples


Zero-shot comparison captions: 100%|██████████| 4/4 [00:05<00:00,  1.28s/it]

01:49:00 |  366 | INFO     | Running zero-shot caption generation for metric samples



Zero-shot metric captions: 100%|██████████| 35/35 [00:44<00:00,  1.27s/it]


Zero-shot scores: {'bleu': 0.0, 'meteor': 0.1032071688734147, 'cider_d': 0.023962586154704844}
Zero-shot predictions saved.


## zero-shot model 메모리 정리

baseline caption을 만든 뒤 base model 메모리를 비워 rank별 LoRA 학습 공간을 확보한다.

 = 더 이상 필요없는 zero-shot 모델을 GPU 메모리에서 제거하는 코드이다. (그냥 두면 GPU 메모리를 계속 차지하기 때문)


In [10]:
try:
    del zero_shot_model
except:
    pass

try:
    del zero_shot_bundle
except:
    pass

import gc
gc.collect()

torch.cuda.empty_cache()

print("memory cleanup done")

memory cleanup done


## LoRA 학습 실행

설정된 rank 목록을 순회하며 LoRA 학습, 평가, artifact 저장을 실행한다. 기본값은 rank 4/8/16 sweep이다.


In [ ]:
# LoRA 학습 실행

rank_summary_rows = []
rank_results = []
full_finetuning_trainable_params = None

# zero-shot 모델 메모리 해제: 이미 삭제되어 있어도 에러 안 나게 처리
try:
    del zero_shot_model
except NameError:
    pass

try:
    del zero_shot_bundle["model"]
except Exception:
    pass

torch.cuda.empty_cache()
release_cuda_memory(device, torch)

for rank in config_lora_ft.ranks:
    rank_result = train_and_evaluate_lora_rank(
        rank=int(rank),
        config=config_lora_ft,
        processor=processor,
        auto_model_class=auto_model_class,
        transformers_module=transformers,
        torch_module=torch,
        torch_dtype=torch.float16,   # TITAN Xp에서는 bfloat16보다 float16 권장
        device=device,
        train_rows=train_rows,
        val_rows=val_rows,
        comparison_samples=comparison_samples,
        metric_samples=metric_samples,
        zero_shot_comparison_predictions=zero_shot_comparison_predictions,
        zero_shot_metric_predictions=zero_shot_metric_predictions,
        full_finetuning_trainable_params=full_finetuning_trainable_params,
        logger=LOGGER,
    )

    full_finetuning_trainable_params = rank_result["full_finetuning_trainable_params"]
    rank_summary_rows.append(rank_result["summary"])
    rank_results.append(rank_result)

01:49:53 |   75 | INFO     | Starting LoRA fine-tuning | rank=4 train_rows=28 val_rows=7 effective_batch=4 estimated_optimizer_steps=70


Epoch,Training Loss,Validation Loss
1,4.185100,4.135925
2,2.699100,3.018583


## rank summary와 method comparison 저장

LoRA rank별 결과와 `Zero-shot / Full FT / LoRA` 비교 table을 함께 저장한다.

- full fine-tuning 산출물이 없으면 skip reason을 남기고 LoRA 결과는 그대로 저장한다.





> rank summary와 method comparizon 저장 코드

이 코드는 모든 학습 결과를 한 곳에 모아서 정리하고 저장하는 코드이다.

> rank_summary, method_comparison

```
rank_summary.json      → 모든 정보 (종합)
method_comparison.json → 비교 결과만 (간단)
```







In [ ]:
# rank summary와 zero-shot / LoRA 비교 결과 저장
# Full Fine-tuning 결과는 사용하지 않고, Zero-shot과 LoRA 결과만 비교한다.

config_payload = build_config_payload(config_lora_ft)

# rank_results 안에는 rank별 before_after_report, caption_scores 등이 들어있음
method_comparison_report = {
    "zero_shot_caption_scores": zero_shot_scores,
    "lora_results": rank_results,
}

# 전체 요약 저장
save_json(
    config_lora_ft.output_dir / "rank_summary.json",
    {
        "config": config_payload,
        "zero_shot_caption_scores": zero_shot_scores,
        "rows": rank_summary_rows,
        "method_comparison_report": method_comparison_report,
    },
)

# 비교 결과만 따로 저장
save_json(
    config_lora_ft.output_dir / "method_comparison.json",
    method_comparison_report,
)

# 결과 출력
print("Rank summary rows:")
for row in rank_summary_rows:
    print(row)

result_lora_ft = {
    "config": config_payload,
    "zero_shot_caption_scores": zero_shot_scores,
    "rows": rank_summary_rows,
    "method_comparison_report": method_comparison_report,
}

print("Saved:")
print(config_lora_ft.output_dir / "rank_summary.json")
print(config_lora_ft.output_dir / "method_comparison.json")

## LoRA fine-tuning 결과 시각화

`show_lora_rank_summary`로 zero-shot, full fine-tuning, rank별 LoRA 결과를 같은 metric 축에서 비교한다. full fine-tuning 산출물이 없거나 metric을 읽지 못한 경우에는 해당 항목을 표시하지 않는다.

* 아래 코드는 zero-shot, Full FT, LoRA Rank별 결과를 그래프로 그리는 코드이다.



> zero-shot 점수 꺼낼때


```
계산 성공 → zero_shot_plot_metrics = {"bleu": 0.21, ...}  ← 그래프에 표시
계산 실패 → zero_shot_plot_metrics = None                 ← 그래프에서 제외
```




In [ ]:
from utils.visualization import show_lora_rank_summary

# LoRA fine-tuning 결과 시각화
# Zero-shot과 LoRA rank 결과만 비교한다.

zero_shot_plot_metrics = result_lora_ft["zero_shot_caption_scores"]

if isinstance(zero_shot_plot_metrics, dict) and zero_shot_plot_metrics.get("available") is False:
    zero_shot_plot_metrics = None

show_lora_rank_summary(
    result_lora_ft["rows"],
    zero_shot_metrics=zero_shot_plot_metrics,
    full_finetuning_metrics=None,
)